In [ ]:
from src.model import Scheduler ,MiniGpt
from src.dataset import get_batch
import torch
import os


torch.manual_seed(42)


# 1. Initialize a small model
model = MiniGpt(
    vocab_size=100,
    d_model=64,
    context_size=32,
    num_heads=4,
    d_ff=256,
    num_layers=2
)

# 2. Initialize optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

# 3. Initialize scheduler
scheduler = Scheduler(
    optimizer,
    warmupiter=2,
    maxiter=10
)

# 4. Generate synthetic token data
train_data = torch.randint(0, 100, (5000,))
val_data = torch.randint(0, 100, (1000,))


x, y = get_batch(train_data, 4, 16)

model.eval()
with torch.no_grad():
    logits, initial_loss = model(x, y)

print("Initial loss:", initial_loss.item())
print("Logits standard deviation:", logits.std().item())

print(
    "Token embedding std:",
    model.transformer.Embeddings.weight.std().item()
)

print(
    "Position embedding std:",
    model.transformer.PositionalEmbeddings.weight.std().item()
)

# 5. Run 10 training steps
train_losses, val_losses = model.train_model(
    optimizer,
    train_data,
    val_data,
    eval_interval=5
    eval_steps=2,
    schedular=scheduler,
    batch_size=4,
    block_size=16,
    max_steps=10
)

# 6. Verify results
assert len(train_losses) == 10
assert len(val_losses) == 2

assert all(
    torch.isfinite(torch.tensor(train_losses))
)

assert os.path.isfile("./checkpoints/best_model.pt")

print("Training pipeline passed!")
print("Final training loss:", train_losses[-1])
print("Validation losses:", val_losses)


Initial loss: 4.607849597930908
Logits standard deviation: 0.1571214646100998
Token embedding std: 0.019994493573904037
Position embedding std: 0.02049083262681961
validation loss is 4.646124601364136
train Loss is 4.615816116333008
validation loss is 4.61464262008667
Training pipeline passed!
Final training loss: 4.615816116333008
Validation losses: [4.646124601364136, 4.61464262008667]


In [2]:

from src.model import MiniGpt, Scheduler, load_checkpoint
import torch

# 1. Create a fresh model
restored_model = MiniGpt(
    vocab_size=100,
    d_model=64,
    context_size=32,
    num_heads=4,
    d_ff=256,
    num_layers=2
)

# 2. Create a fresh optimizer
restored_optimizer = torch.optim.AdamW(
    restored_model.parameters(),
    lr=3e-4
)

# 3. Create the same scheduler
restored_scheduler = Scheduler(
    restored_optimizer,
    warmupiter=2,
    maxiter=10
)

# 4. Load saved training state
step, val_loss = load_checkpoint(
    restored_model,
    restored_optimizer,
    restored_scheduler,
    "./checkpoints/best_model.pt"
)

print("Restored step:", step)
print("Restored validation loss:", val_loss)
print(
    "Restored learning rate:",
    restored_optimizer.param_groups[0]["lr"]
)

# 5. Verify weight tying
assert (
    restored_model.modelhead.weight
    is restored_model.transformer.Embeddings.weight
)

# 6. Compare original and restored model
model.eval()
restored_model.eval()

test_tokens = torch.randint(0, 100, (2, 16))

with torch.no_grad():
    original_logits, _ = model(test_tokens)
    restored_logits, _ = restored_model(test_tokens)

print(
    "Maximum logits difference:",
    (original_logits - restored_logits).abs().max().item()
)


Restored step: 10
Restored validation loss: 4.61464262008667
Restored learning rate: 3e-05
Maximum logits difference: 0.0


In [4]:
from scripts.prepare_data import LoadData, tokenize

DATASET_ID = "skeskinen/TinyStories-hf"
train_stories, val_stories = LoadData(DATASET_ID)

print("Training stories:", len(train_stories))
print("Validation stories:", len(val_stories))


Training stories: 9000
Validation stories: 1000


In [7]:
import tiktoken

# Tokenize both datasets
train_data = tokenize(train_stories)
val_data = tokenize(val_stories)

# Verify tensor properties
print("Training tokens:", train_data.numel())
print("Validation tokens:", val_data.numel())
print("Training shape:", train_data.shape)
print("Validation shape:", val_data.shape)
print("Data type:", train_data.dtype)

# Verify token ID bounds
print("Minimum token ID:", train_data.min().item())
print("Maximum token ID:", train_data.max().item())

# Verify EOS tokens
tokenizer = tiktoken.get_encoding("gpt2")
eos_id = tokenizer.eot_token

print("Training EOS count:", (train_data == eos_id).sum().item())
print("Validation EOS count:", (val_data == eos_id).sum().item())

assert train_data.ndim == val_data.ndim == 1
assert train_data.dtype == val_data.dtype == torch.long
assert train_data.min().item() >= 0
assert train_data.max().item() < 50257
assert val_data.min().item() >= 0
assert val_data.max().item() < 50257
assert (train_data == eos_id).sum().item() == 9000
assert (val_data == eos_id).sum().item() == 1000

print("Tokenization test passed!")


Training tokens: 1927748
Validation tokens: 194559
Training shape: torch.Size([1927748])
Validation shape: torch.Size([194559])
Data type: torch.int64
Minimum token ID: 0
Maximum token ID: 50256
Training EOS count: 9000
Validation EOS count: 1000
Tokenization test passed!


In [8]:

import torch
from src.dataset import get_batch

batch_size = 16
block_size = 128

x, y = get_batch(train_data, batch_size, block_size)

print("Input shape:", x.shape)
print("Target shape:", y.shape)

# Verify next-token prediction alignment
assert x.shape == (16, 128)
assert y.shape == (16, 128)
assert torch.equal(x[:, 1:], y[:, :-1])

print("Batch test passed!")


Input shape: torch.Size([16, 128])
Target shape: torch.Size([16, 128])
Batch test passed!


In [9]:
from pathlib import Path
output_directory=Path("data/processed")
output_directory.mkdir(parents=True , exist_ok=True)
torch.save(train_data,output_directory/"train_data.pt")
torch.save(val_data,output_directory/"val_data.pt")